<!-- ---
title: Deploy NVIDIA Nemotron 3 Nano Omni on Microsoft Foundry: Unifying video, audio, image and text understanding.
author: Juan Julián Cea Morán
--- -->

# Deploy NVIDIA Nemotron 3 Nano Omni on Microsoft Foundry: Unifying video, audio, image and text understanding.

In this example, you will deploy [`nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16`](https://huggingface.co/microsoft/nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16) on Microsoft Foundry and then use it to answer questions about audio, video and image content.

**NVIDIA Nemotron 3 Nano Omni** is a multimodal large language model that unifies video, audio, image, and text understanding into a single, highly efficient open model. Built to replace fragmented vision‑language‑audio stacks, it extends the Nemotron Nano family with integrated video+speech comprehension, Graphical User Interface (GUI), Optical Character Recognition (OCR), and speech transcription capabilities, enabling enterprise-grade Q&A, summarization, transcription, and document intelligence workflows.

![NVIDIA Nemotron 3 Nano Omni on Microsoft Foundry](./nemotron-3-nano-omni-microsoft-foundry.png)

The Nemotron 3 Nano Omni architecture extends Nemotron 3 Nano by bringing multimodal perception and reasoning into a single 30B hybrid MoE model.

- **🧠 Hybrid MoE architecture**: Combines Mamba layers for sequence and memory efficiency with transformer layers for precise reasoning.
- **📷 Spatiotemporal visual processing and efficient video sampling**: The inference-time Efficient Video Sampling (EVS) layer compresses the high-density visual tokens from multiple frames into a concise set that the LLM can process without overwhelming its context window.
- **🎨 Multimodal architecture**: 
    - A strong text model at its core, preserving the foundation model’s language ability.
    - Audio integration built upon the NVIDIA Parakeet encoder and specialized datasets that move beyond simple transcription.
    - C-RADIOv4-H and Encoder-based Video Summarization to handle high-resolution images and dynamic video.


![NVIDIA Nemotron 3 Nano Omni Workflow](./nemotron-3-nano-omni-workflow.png)


For more information, make sure to check [the model card on the Hugging Face Hub](https://huggingface.co/nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16), or the official [NVIDIA blog post](https://developer.nvidia.com/blog/nvidia-nemotron-3-nano-omni-powers-multimodal-agent-reasoning-in-a-single-efficient-open-model/).

## Requirements

To run this example, you need to meet the following prerequisites. You can also read more about them in the [Azure Machine Learning Tutorial: Create resources you need to get started](https://learn.microsoft.com/en-us/azure/machine-learning/quickstart-create-resources?view=azureml-api-2).

- You have a Microsoft Azure subscription and are logged in
- You have the `az` CLI installed
- You have the necessary permissions to:
    - Create an Azure Machine Learning Managed Online Endpoint
    - Create an Azure Machine Learning Deployment
- You have Python 3.10+ installed locally and `pip`

For more information, please go through the steps in the guide ["Configure Azure Machine Learning and Microsoft Foundry"](https://huggingface.co/docs/microsoft-azure/guides/configure-azure-ml-microsoft-foundry).

## Setup

### Set environment variables

For convenience, you can set the following environment variables to be used throughout the example:

In [ ]:
%env LOCATION eastus
%env SUBSCRIPTION_ID <YOUR_SUBSCRIPTION_ID>
%env RESOURCE_GROUP <YOUR_RESOURCE_GROUP>
%env WORKSPACE_NAME <YOUR_WORKSPACE_NAME>

In [7]:
import os
from uuid import uuid4

os.environ["ENDPOINT_NAME"] = f"nemotron-omni-{str(uuid4())[:8]}"
os.environ["DEPLOYMENT_NAME"] = f"nemotron-omni-{str(uuid4())[:8]}"

### Install Azure Python SDK (+ dependencies)

You need to install some Azure Python SDK dependencies:

* `azure-identity` to use the `DefaultAzureCredential` authentication with your Managed Identity
* `azure-ai-ml` to create the Azure Machine Learning Managed Online Endpoint + Deployment, and to invoke it

In [3]:
%pip install azure-ai-ml azure-identity --upgrade --quiet


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Authenticate to Azure Machine Learning

You can then authenticate to Azure Machine Learning with your Managed Identity with Python as:

In [ ]:
import os
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=os.getenv("SUBSCRIPTION_ID"),
    resource_group_name=os.getenv("RESOURCE_GROUP"),
    workspace_name=os.getenv("WORKSPACE_NAME"),
)

## Create endpoint + deployment

### Create endpoint

Now you need to create the [ManagedOnlineEndpoint via the Azure Machine Learning Python SDK](https://learn.microsoft.com/en-us/python/api/azure-ai-ml/azure.ai.ml.entities.managedonlineendpoint?view=azure-python) as follows:

> [!NOTE]
> Every model in the Hugging Face collection is powered by an efficient inference backend, and each of those can run on a wide variety of instance types (as listed in [Supported Hardware](https://huggingface.co/docs/microsoft-azure/azure-ai/supported-hardware)). Since models and inference engines require a GPU-accelerated instance, you might need to request a quota increase as per [Manage and increase quotas and limits for resources with Azure Machine Learning](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-manage-quotas?view=azureml-api-2).

In [9]:
from azure.ai.ml.entities import ManagedOnlineEndpoint

endpoint = ManagedOnlineEndpoint(name=os.getenv("ENDPOINT_NAME"))
client.begin_create_or_update(endpoint).wait()

### Create deployment

Once the endpoint is created, you need to create the deployment indicating which model, hardware, and settings to use. In this case, we will be using a single NVIDIA H100, enough for deploying our model. Also, we'll be increasing the default timeout per request to 180s to ensure that we can process long image and audio files in time.

In [10]:
from azure.ai.ml.entities import ManagedOnlineDeployment, OnlineRequestSettings

deployment = ManagedOnlineDeployment(
    name=os.getenv("DEPLOYMENT_NAME"),
    endpoint_name=os.getenv("ENDPOINT_NAME"),
    model=f"azureml://registries/HuggingFace/models/{os.getenv('MODEL_ID').replace('/', '-').replace('_', '-').lower()}/labels/latest",
    instance_type="Standard_ND96amsr_A100_v4",
    instance_count=1,
    request_settings=OnlineRequestSettings(request_timeout_ms=180000),
)
client.online_deployments.begin_create_or_update(deployment).result()

Check: endpoint nemotron-omni-d314e939 exists


.................................................................................................................................................................................................................................................

ManagedOnlineDeployment({'private_network_connection': None, 'package_model': False, 'provisioning_state': 'Succeeded', 'endpoint_name': 'nemotron-omni-d314e939', 'type': 'Managed', 'name': 'nemotron-omni-4c289ae2', 'description': None, 'tags': {}, 'properties': {'AzureAsyncOperationUri': 'https://management.azure.com/subscriptions/96f8b384-0587-41d4-9105-9fe6dca745b3/providers/Microsoft.MachineLearningServices/locations/uksouth/mfeOperationsStatus/odidp:787817d7-65a1-411b-82f0-09d3ab08460a:fb391935-4d85-44e7-a3cc-69a193c32b66?api-version=2023-04-01-preview'}, 'print_as_yaml': False, 'id': '/subscriptions/96f8b384-0587-41d4-9105-9fe6dca745b3/resourceGroups/rg-juanju-project/providers/Microsoft.MachineLearningServices/workspaces/juanmoran-4413/onlineEndpoints/nemotron-omni-d314e939/deployments/nemotron-omni-4c289ae2', 'Resource__source_path': '', 'base_path': '/Users/juanjucm/Desktop/projects/Microsoft-Azure/examples/foundry/deploy-nemotron-3-nano-omni', 'creation_context': <azure.ai.ml

![Azure AI Deployment from Azure AI Foundry](./azure-ai-deployment.png)

> [!WARNING]
> The deployment might take ~30 minutes, but it could also take longer depending on the selected SKU availability in the region.

## Run inference on the Foundry Endpoint

Now that the Foundry Endpoint is up and running, we are ready to run inference on it. We will be using the **OpenAI Python SDK** to send requests to our endpoint, which is by served through an **OpenAI-compatible Chat Completions API** at `/v1/chat/completions`.

For this example, we will query the model using all the accepted input modalities (e.g., audio, image, and video). Additionally, we will test the model with a mix of audio and image in the same prompt to see how it performs in multimodality scenarios.

### Set up the OpenAI Python SDK

In [11]:
%pip install openai --upgrade --quiet


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Retrieve the endpoint URL and API key, then create the OpenAI client, making sure to include the `azureml-model-deployment` header so every request is routed to the right deployment. Setting it once via `default_headers` is the recommended approach since the header needs to accompany each request.

In [12]:
from urllib.parse import urlsplit
from openai import OpenAI
import os

api_key = client.online_endpoints.get_keys(os.getenv("ENDPOINT_NAME")).primary_key
url_parts = urlsplit(client.online_endpoints.get(os.getenv("ENDPOINT_NAME")).scoring_uri)
api_url = f"{url_parts.scheme}://{url_parts.netloc}/v1"

openai_client = OpenAI(
    base_url=api_url,
    api_key=api_key,
    default_headers={"azureml-model-deployment": os.getenv("DEPLOYMENT_NAME")},
)

> [!NOTE]
> Alternatively, you can also build the API URL manually as follows, since the URIs are globally unique per region, meaning that there will only be one endpoint named the same way within the same region:
> ```python
> api_url = f"https://{os.getenv('ENDPOINT_NAME')}.{os.getenv('LOCATION')}.inference.ml.azure.com/v1"
> ```
> Or just retrieve it from either Microsoft Foundry or the Azure Machine Learning Studio.

### Image understanding: Art historian AI tutor

In this first example, we will use Nemotron's image capabilities to create an AI tutor for art history. We will send an image of the famous painting "Guernica" by Pablo Picasso, and prompt the model to generate Q&A pairs about it.

![Azure AI Deployment from Azure AI Foundry](./guernica.jpeg)

In [ ]:

image_url = "https://recursos.museoreinasofia.es/styles/large_landscape/public/Obra/DE00050_2.jpg.webp"

response = openai_client.chat.completions.create(
    model=os.getenv("MODEL_ID"),
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": image_url
                        }
                },
                {"type": "text",
                 "text": "You are a high school history teacher.\ "
                 "Given this image, generate 5 high-school level questions and answers covering authorship, artistic movement, historical context and symbolism."},
            ],
        }
    ],
)
print(response.choices[0].message.content)



**1. Who created the painting “Guernica,” and in what year was it completed?**  
*Answer:* The painting was created by the Spanish artist **Pablo Picasso** in **1937**.

---

**2. Which artistic movement is “Guernica” most closely associated with, and what are two key characteristics of that movement visible in the work?**  
*Answer:* “Guernica” is a seminal work of **Cubism** (specifically, its later, more politically charged form). Two visible characteristics are:  
- **Fragmented, geometric forms** that break the figures and objects into angular planes, and  
- **Limited, monochromatic palette** (black, white, and shades of gray) that emphasizes shape and emotion rather than realistic color.

---

**3. What historical event does “Guenica” depict, and why was Picasso commissioned to create it?**  
*Answer:* The painting portrays the **bombing of the Basque town of Guernica** during the Spanish Civil War (April 26 1937) by German and Italian fascist aircraft at the request of the Nat

### Video understanding: Generating a cookbook recipe from a YouTube video

For showcasing the video understanding capabilities, we will download a YouTube video of a [tiramisu recipe by El Comidista](https://www.youtube.com/watch?v=tG8NNobawj0), and ask the model to generate a cuisine-book style recipe based on it. The model will have to use both the visual and the audio information from the video to generate and accurate recipe.

In [41]:
%pip install yt_dlp --upgrade --quiet


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [42]:
import yt_dlp

youtube_url = "https://www.youtube.com/watch?v=tG8NNobawj0"

ydl_opts = {
    "format": "bv*[ext=mp4]+ba[ext=m4a]/b[ext=mp4]/best",
    "merge_output_format": "mp4",
    "outtmpl": "input.%(ext)s",
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download([youtube_url])

[youtube] Extracting URL: https://www.youtube.com/watch?v=tG8NNobawj0
[youtube] tG8NNobawj0: Downloading webpage


[youtube] tG8NNobawj0: Downloading android vr player API JSON
[info] tG8NNobawj0: Downloading 1 format(s): 136+140
[download] input.mp4 has already been downloaded


In [ ]:

from pathlib import Path
import base64
import mimetypes

# Load video, convert to base64 and prepare the payload.
video = Path("./input.mp4")
with open(video, "rb") as f:
    mime_type = mimetypes.guess_type(video)[0] or "video/mp4"
    video_b64 = base64.b64encode(f.read()).decode("utf-8")
    video_url = f"data:{mime_type};base64,{video_b64}"
 
response = openai_client.chat.completions.create(
    model=os.getenv("MODEL_ID"),
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "video_url",
                    "video_url": {
                        "url": video_url
                    }
                },
                {"type": "text", "text": "Write a cuisine-book style recipe in English based on this video. \
                    Start with a short introduction for the recipe translating the speaker's words.\
                    After that, add a section listing all the ingredients that appear in the video. \
                    Finally, explain the preparation process in bullet points, sumarizing the instructions in the video."},
            ],
        }
    ],
    max_tokens=20480,
    temperature=0.4,
    extra_body={
        "mm_processor_kwargs": {"use_audio_in_video": True},
    },
)
print(response.choices[0].message.content)



**A Light, Beer‑Infused Tiramisu**  
A fresh, slightly less sweet take on the classic tiramisu, made with black beer instead of coffee. This airy dessert is quick to assemble, needs only a couple of hours in the fridge, and delivers a delightful balance of creamy richness and subtle maltiness.

**Ingredients**  
- 3 large eggs  
- 6 tablespoons granulated sugar (plus a little extra for the mascarpone)  
- 200 g mascarpone cheese  
- 150 ml black beer (cerveza negra)  
- Ladyfinger biscuits (about 12–15 pieces)  
- Grated dark chocolate (for garnish)  
- Pinch of salt  

**Preparation**  

- **Separate the eggs:** Crack the eggs and place the whites in one bowl and the yolks in another.  
- **Whip the whites:** Beat the egg whites with a pinch of salt until they form stiff peaks. Cover the bowl and refrigerate while you prepare the rest.  
- **Mix yolks and sugar:** In the yolk bowl, whisk the 6 tablespoons of sugar into the yolks until the mixture is pale, thick, and creamy.  
- **Com

### Multimodal capabilities: Image and audio fact cheking

## Release resources

Once you are done using the Foundry Endpoint, you can delete the resources (i.e., you will stop paying for the instance on which the model is running and all the attached costs) as follows:

In [ ]:
client.online_endpoints.begin_delete(name=os.getenv("ENDPOINT_NAME")).result()

## Conclusion

Throughout this example, you learned how to deploy `nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16` as an Azure Machine Learning Managed Online Endpoint on Microsoft Foundry, and how to exercise its key inference capabilities: multimodal understanding of images, videos, and audio, as well as standard text-based inference.

If you have any doubt, issue or question about this example, feel free to [open an issue](https://github.com/huggingface/Microsoft-Azure/issues/new) and we'll do our best to help!